# SOASEC — training finale: dati completi + modello corretto

Addestra i due modelli (con e senza tempo) su **tutto** il training split, con
le **due correzioni** al modello attive.

## Le due correzioni

| opzione | cosa era rotto | effetto misurato |
|---|---|---|
| `--edge-inputs` | la libreria non passava `time` ed `edge_attr` all'attenzione (da PyG 2.5 il controllo sulla firma di `propagate` fallisce in silenzio) | cambiando **tutti** i Δt o **tutti** i tipi d'arco, l'output cambiava di **0,000000** |
| `--self-loops` | il `forward` della libreria non aggiunge i self-loop: una casella non vede le proprie feature | nel 97% dei pareggi le caselle di destinazione avevano embedding identiche al bit |

Senza le opzioni il modello si comporta esattamente come prima: i risultati
della v1 restano riproducibili. Le due correzioni valgono per **entrambi** i
modelli, quindi il confronto con/senza tempo resta pulito.

## I dati

| | v1 | **questo** |
|---|---|---|
| esempi | 470.342 | **~2.590.000** |
| % del training | 13% | **100%** |
| epoche | 50 | **10** |
| presentazioni (esempi x epoche) | 23,5 M | **25,9 M** |
| minuti/epoca | 5,9 | **~35** |

Stesso budget di calcolo della v1, ma dati 5,5 volte piu' vari e con la stessa
distribuzione per profondita' del test.

## Come e' fatto questo notebook

- **Ogni comando mostra il suo output riga per riga** e, se fallisce, **ferma
  il notebook** con un errore chiaro invece di andare avanti.
- **Se interrompi una cella**, il processo che stava girando viene fermato
  davvero (niente training fantasma in background).
- I dataset **si ripristinano da Drive** se ci sono gia'; altrimenti si
  costruiscono (~50 minuti l'uno) e si archiviano.

> **Runtime → Cambia tipo di runtime → GPU** prima di eseguire, celle in ordine.


---
## 1. Drive e installazione


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
DRIVE = '/content/drive/MyDrive/SOASEC'
assert os.path.isdir(DRIVE), f'non trovo {DRIVE}'

need = ['graphs.zip', 'TimeGNN-main.zip', 'train.csv']
have = set(os.listdir(DRIVE))
print('file necessari:')
for f in need:
    print(('  OK    ' if f in have else '  MANCA ') + f)
assert all(f in have for f in need), 'carica i file mancanti su Drive'

print('\ndataset gia archiviati (se mancano, verranno costruiti: ~50 min l\'uno):')
for nome in ('train_full_timing', 'train_full_plain'):
    tar, man = f'{nome}.tar', f'{nome}.manifest.json'
    if tar in have and man in have:
        stato = 'pronto'
    elif tar in have:
        stato = 'archivio senza manifest: verra\' controllato'
    else:
        stato = 'da costruire'
    print(f'  {nome:<20} {stato}')

tot, used, free = shutil.disk_usage('/content/drive')
print(f'\nDrive: circa {free / 1e9:.1f} GB liberi')

In [ ]:
!cp '{DRIVE}/TimeGNN-main.zip' /content/ && cd /content && unzip -q -o TimeGNN-main.zip

import numpy
NP = numpy.__version__   # numpy bloccato: e' il suo aggiornamento che rompe torch
!pip install -q torch_geometric chess seaborn matplotlib 'numpy=={NP}'
print('fatto, ora riavvio il runtime')

## 2. Riavvio

Vedrai *"Your session crashed"*: e' voluto. `pip` ha toccato librerie che torch
aveva gia' caricato; senza riavvio si rompe con `module functions cannot set
METH_CLASS`.

Dopo il riavvio **riparti dalla sezione 3**.


In [ ]:
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

---
## 3. Preparazione

*(da qui si riparte dopo il riavvio, e anche dopo un crash del runtime)*

La prima cella definisce gli strumenti usati da tutte le altre: se dopo un
riavvio una cella dice `NameError: name 'sh' is not defined`, riesegui questa.


In [ ]:
import os, sys, glob, json, time, signal, site, subprocess

DRIVE = '/content/drive/MyDrive/SOASEC'
REPO  = 'https://github.com/FaiellaFrancesco/Progetto-SOASEC.git'
SRC   = '/content/Progetto-SOASEC/src/dataset_scripts'
CSV   = '/content/train.csv'

if not os.path.isdir(DRIVE):
    from google.colab import drive
    drive.mount('/content/drive')


def sh(cmd, log=None):
    """INPUT : a shell command, and optionally a file to copy its output to
       OUTPUT: none; raises if the command fails

    Colab does not show the output of subprocess.run() in the cell, so the
    output is read line by line and printed here. If the cell is interrupted,
    the whole process group is killed: otherwise a training would keep running
    in the background, and relaunching would start a second one on top."""
    out = open(log, 'a') if log else None
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1,
                         start_new_session=True)
    try:
        for line in p.stdout:
            print(line, end='', flush=True)
            if out:
                out.write(line)
                out.flush()
        rc = p.wait()
    except KeyboardInterrupt:
        os.killpg(p.pid, signal.SIGTERM)
        p.wait()
        print('\n>>> interrotto: processo fermato')
        raise
    finally:
        if out:
            out.close()
    if rc != 0:
        raise RuntimeError(f'comando fallito (codice {rc}):\n{cmd}')


import torch
assert torch.cuda.is_available(), \
    'nessuna GPU: Runtime -> Cambia tipo di runtime -> GPU, poi ricomincia'
print('GPU:', torch.cuda.get_device_name(0))
print('strumenti pronti')

In [ ]:
t0 = time.time()
if not os.path.isdir('/content/graphs/val_timing'):
    sh(f"cp '{DRIVE}/graphs.zip' /content/ && cd /content && unzip -q -o graphs.zip")
if not os.path.isfile(CSV):
    sh(f"cp '{DRIVE}/train.csv' /content/")
if not os.path.isdir('/content/TimeGNN-main'):
    sh(f"cp '{DRIVE}/TimeGNN-main.zip' /content/ && cd /content && unzip -q -o TimeGNN-main.zip")

# il codice si prende SEMPRE fresco da GitHub: cosi' ci sono le correzioni
sh('rm -rf /content/Progetto-SOASEC')
sh(f'git clone -q -b faiella {REPO} /content/Progetto-SOASEC')
sh('cd /content/Progetto-SOASEC && git log --oneline -1')
print(f'dati e codice pronti in {time.time() - t0:.0f} s')

src = open(f'{SRC}/model.py').read()
assert 'restore_edge_inputs' in src and 'with_self_loops' in src, \
    'model.py NON contiene le correzioni: hai fatto git push dal Mac?'
assert os.path.isfile(f'{SRC}/test_model_v2.py'), \
    'manca test_model_v2.py su GitHub: git add test_model_v2.py, commit, push'
print('model.py contiene le due correzioni')

In [ ]:
# import che funzionano da qualunque processo, senza dipendere da PYTHONPATH
cand = glob.glob('/content/**/timegnn/__init__.py', recursive=True)
assert cand, 'timegnn non trovata'
LIB = os.path.dirname(os.path.dirname(cand[0]))
for p in (SRC, LIB):
    if p not in sys.path:
        sys.path.insert(0, p)
os.environ['PYTHONPATH'] = f'{SRC}:{LIB}'
with open(os.path.join(site.getsitepackages()[0], 'soasec_paths.pth'), 'w') as f:
    f.write(LIB + '\n' + SRC + '\n')

env = {k: v for k, v in os.environ.items() if k != 'PYTHONPATH'}
r = subprocess.run([sys.executable, '-c', 'import timegnn, model, graph_dataset'],
                   cwd='/', env=env, capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-800:]
print('import OK da qualunque processo')

---
## 4. I due dataset

Per ciascuno, nell'ordine meno costoso:

1. gia' in `/content` → niente
2. archiviato su Drive → lo scompatta e **verifica** il numero di file
3. altrimenti → lo costruisce (~50 min) e lo archivia su Drive

Un archivio senza manifest (scrittura interrotta, o creato da una versione
precedente del notebook) viene controllato leggendolo per intero: se e' sano
si usa, se e' troncato si ricostruisce.


In [ ]:
os.makedirs('/content/graphs', exist_ok=True)


def prepara(nome, flag):
    """INPUT : dataset name, and '--timing' or '' for the plain one
       OUTPUT: the parquet files exist under /content/graphs/<nome>"""
    out = f'/content/graphs/{nome}'
    tar = f'{DRIVE}/{nome}.tar'
    man = f'{DRIVE}/{nome}.manifest.json'

    n = len(glob.glob(f'{out}/*.parquet'))
    if n:
        print(f'{nome}: gia in /content ({n} parquet)')
        return

    if os.path.isfile(tar) and not os.path.isfile(man):
        print(f'{nome}: archivio senza manifest, lo controllo (1-2 min)...')
        r = subprocess.run(['tar', '-tf', tar], capture_output=True, text=True)
        if r.returncode == 0:
            k = sum(1 for l in r.stdout.splitlines() if l.endswith('.parquet'))
            json.dump({'n_files': k}, open(man, 'w'))
            print(f'  sano, {k} file: manifest scritto')
        else:
            print('  troncato: lo cancello e ricostruisco')
            os.remove(tar)

    if os.path.isfile(tar) and os.path.isfile(man):
        atteso = json.load(open(man))['n_files']
        t0 = time.time()
        sh(f"tar -xf '{tar}' -C /content/graphs")
        n = len(glob.glob(f'{out}/*.parquet'))
        assert n == atteso, (f'{nome}: {n} file invece di {atteso}. Cancella '
                             f'{tar} e {man} da Drive e rilancia')
        print(f'{nome}: ripristinato da Drive in {time.time()-t0:.0f} s '
              f'({n} parquet, verificati)')
        return

    print(f'\n=== {nome}: costruzione (~50 min) ===', flush=True)
    t0 = time.time()
    sh(f'cd {SRC} && python -u db_rows_to_graphs.py -i {CSV} -o {out} {flag}')
    print(f'build: {(time.time()-t0)/60:.1f} min')
    t0 = time.time()
    sh(f"tar -cf '{tar}' -C /content/graphs {nome}")
    n = len(glob.glob(f'{out}/*.parquet'))
    json.dump({'n_files': n}, open(man, 'w'))   # per ULTIMO: tar completo
    print(f'{nome}: archiviato su Drive, {n} file, {time.time()-t0:.0f} s')


prepara('train_full_timing', '--timing')
prepara('train_full_plain', '')

In [ ]:
# stesso numero di esempi nei due dataset, altrimenti l'ablation non vale
import pandas as pd
n = {}
for nome in ('train_full_timing', 'train_full_plain'):
    files = glob.glob(f'/content/graphs/{nome}/*.parquet')
    n[nome] = sum(len(pd.read_parquet(f, columns=['y'])) for f in files)
    print(f'{nome:<20} {n[nome]:>11,} esempi   ({len(files)} file)')
assert n['train_full_timing'] == n['train_full_plain'], \
    'numero di esempi diverso: ablation compromessa'
assert 2_400_000 < n['train_full_timing'] < 2_800_000, \
    'numero di esempi lontano dai ~2.590.000 attesi: controlla train.csv'
print('\nOK: allineati, e nell\'ordine di grandezza atteso')

---
## 5. Controllo pre-volo

Se un test fallisce la cella si ferma con un errore, e il training non va
lanciato.

- `test_model_v2.py` deve dire **14/14**. La sua parte **`0. diagnosi`** dice
  se in QUESTO ambiente (la stessa versione di PyG con cui e' stata addestrata
  la v1) il modello originale ignorava tempo e tipi d'arco. **Copiala**: serve
  per la relazione.
- `test_model.py` deve dire **8/8**.


In [ ]:
sh(f'cd {SRC} && python -u test_model_v2.py -i /content/graphs/val_timing')
print()
sh(f'cd {SRC} && python -u test_model.py -i /content/graphs/train_full_timing')

---
## 6. Training CON il tempo

Circa **6 ore**. L'output scorre qui sotto e viene copiato anche in un file.

**Nei primi batch** guarda:

- `loss`: deve partire intorno a **3,3–3,5**. Decine o centinaia = fermati.
- `ex/s`: nei training precedenti erano ~1.300. Qualcosa in meno e' normale (64
  archi in piu' per posizione); sotto 800, fermati e dimmelo.

**Dopo l'epoca 1** (~35 min) guarda la `val acc`. Con il modello vecchio,
un'epoca di questa dimensione avrebbe dato circa **23,5%**. Con le correzioni
ci si aspetta di piu'; se viene sotto, qualcosa non va.

**Se si interrompe:** rilancia questa cella identica, riprende da `last.pt`. Se
e' caduto tutto il runtime, rifai la sezione 3 (poi la 4, che ripristina da
Drive in pochi minuti) e rilancia questa.


In [ ]:
def allena(timing):
    """INPUT : True for the model with time, False for the one without
       OUTPUT: none; trains until the end, resuming from last.pt if present

    The two commands differ ONLY in --timing, the data folders and the output
    folder. Built from one function so they cannot drift apart by a typo."""
    tag = 'timing' if timing else 'plain'
    cmd = (f'cd {SRC} && python -u train.py'
           f' --train /content/graphs/train_full_{tag}'
           f' --val   /content/graphs/val_{tag}'
           f' --out   {DRIVE}/ckpt_fix/{tag}'
           f'{" --timing" if timing else ""}'
           f' --epochs 10 --batch-size 128 --workers 2'
           f' --num-layers 3 --patience 5'
           f' --edge-inputs --self-loops')
    print(cmd, '\n', flush=True)
    sh(cmd, log=f'/content/log_fix_{tag}.txt')


allena(timing=True)

### Andamento

Legge da Drive, quindi funziona anche dopo un riavvio del runtime.


In [ ]:
for cart in ('ckpt_fix/timing', 'ckpt_fix/plain'):
    p = f'{DRIVE}/{cart}/history.json'
    if not os.path.isfile(p):
        print(f'{cart}: ancora nessuna epoca completata')
        continue
    h = json.load(open(p))
    print(f'\n{cart}: {len(h)} epoche')
    print(f"{'epoca':>6} {'train loss':>11} {'val loss':>10} {'val acc':>9}")
    for e in h:
        print(f"{e['epoch']:>6} {e['train_loss']:>11.4f} "
              f"{e['val_loss']:>10.4f} {e['val_acc']:>8.2%}")
    best = min(h, key=lambda e: e['val_loss'])
    print(f"  migliore: epoca {best['epoch']}, val loss {best['val_loss']:.4f}, "
          f"val acc {best['val_acc']:.2%}")

print('\nriferimenti (modello vecchio): v1 migliore val loss 2,5111 in 49 epoche;'
      ' 150k epoca 1 = 20,61%, epoca 2 = 22,95%')

---
## 7. Training SENZA il tempo

**Solo quando il primo e' finito**: una GPU sola, in parallelo si rallentano a
vicenda. Stessa funzione, cambia solo `timing=False`.


In [ ]:
allena(timing=False)

---
## 8. Curve: v1 contro modello corretto

Asse orizzontale in **milioni di esempi visti**: un'epoca della v1 (470 mila
esempi) e un'epoca qui (2,6 milioni) non sono confrontabili.

Colore = modello (blu senza tempo, arancione con tempo); tratto = versione
(tratteggiato v1, continuo corretto).


In [ ]:
import matplotlib.pyplot as plt

BLU, ARANCIO = '#2a78d6', '#eb6834'
INK, INK2, GRID = '#0b0b0b', '#52514e', '#e8e8e4'
serie = [('v1 senza tempo',       'ckpt/plain',      0.470342, BLU,     '--'),
         ('v1 con tempo',         'ckpt/timing',     0.470342, ARANCIO, '--'),
         ('corretto senza tempo', 'ckpt_fix/plain',  2.59,     BLU,     '-'),
         ('corretto con tempo',   'ckpt_fix/timing', 2.59,     ARANCIO, '-')]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.4), facecolor='#fcfcfb')
trovate = 0
for nome, cart, milioni, col, st in serie:
    p = f'{DRIVE}/{cart}/history.json'
    if not os.path.isfile(p):
        print(f'ancora niente per {nome}')
        continue
    h = json.load(open(p))
    trovate += 1
    x = [e['epoch'] * milioni for e in h]       # train.py numera da 1
    a1.plot(x, [e['val_loss'] for e in h], color=col, ls=st, lw=2, label=nome)
    a2.plot(x, [e['val_acc'] for e in h], color=col, ls=st, lw=2, label=nome)

for ax, titolo in ((a1, 'validation loss'), (a2, 'validation accuracy')):
    ax.set_title(titolo, color=INK, fontsize=12, loc='left')
    ax.set_xlabel('milioni di esempi visti', color=INK2, fontsize=10)
    ax.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)
    for s in ('left', 'bottom'):
        ax.spines[s].set_color(GRID)
    ax.tick_params(colors=INK2, labelsize=9)
    ax.set_facecolor('#fcfcfb')
if trovate:
    a1.legend(frameon=False, fontsize=9, labelcolor=INK2)
plt.tight_layout()
plt.show()

---
## 9. Dopo

Su Drive avrai `ckpt_fix/timing/best.pt` e `ckpt_fix/plain/best.pt`. Scaricali
sul Mac in `Progetto-SOASEC/ckpt_fix/timing/` e `Progetto-SOASEC/ckpt_fix/plain/`
(si chiamano uguale: uno per cartella). La valutazione la facciamo insieme:
`load_model` legge le due opzioni dal checkpoint, quindi gli script esistenti
funzionano, ma prima di lanciarli controlliamo che anche `playout_models.py`
costruisca il modello con le opzioni giuste.
